# Connecting Your Agent to External Services with MCP (Model Context Protocol)

In this notebook, you will learn how to connect an AI agent to an **external service** using the **Model Context Protocol (MCP)**. MCP lets your agent reach out to tools hosted on the internet — in this case, the Microsoft Learn server — so it can look up tutorials, documentation, and learning paths on behalf of the user.

By the end, you will have an agent that can answer questions by pulling live information from Microsoft Learn!

## Step 1: Install Packages

First, we install the Python libraries we need. These let us talk to Azure AI Foundry and use the OpenAI-compatible API.

In [ ]:
# Install the required libraries:
# - azure-ai-projects: the SDK for working with Azure AI Foundry (agents, tools, etc.)
# - openai: provides the chat interface we use to talk to our agent
# - python-dotenv: reads secret settings from a .env file
# - azure-identity: handles logging in to Azure automatically
%pip install azure-ai-projects==2.0.0b2 openai==1.109.1 python-dotenv azure-identity

## Step 2: Load Settings

We keep sensitive configuration (like our project endpoint and model name) in a `.env` file. This way, we never accidentally share our secrets. The `load_dotenv()` function loads these values so we can use them in our code.

In [ ]:
# Import everything we need for setup
import os                                          # For reading environment variables
from dotenv import load_dotenv                      # For loading the .env file
from azure.identity import DefaultAzureCredential   # For automatic Azure authentication
from azure.ai.projects import AIProjectClient       # The main Azure AI Foundry client
from azure.ai.projects.models import (              # Classes for defining agents and tools
    PromptAgentDefinition,
    MCPTool,
)

# Read the .env file and load its values into environment variables
load_dotenv()

# my_endpoint: the URL of our Azure AI Foundry project
my_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")

# my_model: the name of the deployed AI model we want the agent to use
my_model = os.getenv("MODEL_DEPLOYMENT_NAME")

## Step 3: Connect to Foundry

We create a **Foundry client** that acts as our gateway to all Azure AI Foundry services. Think of it as opening a connection to the platform.

In [ ]:
# Create the Foundry client
# - endpoint: tells the SDK where our project lives
# - credential: proves we have permission to use it
foundry_client = AIProjectClient(
    endpoint=my_endpoint,
    credential=DefaultAzureCredential(),
)

## Step 4: Create a Chat Client

The **chat client** is an OpenAI-compatible interface that lets us send messages to agents and receive responses. We get it from our Foundry client.

In [ ]:
# Get an OpenAI-compatible chat client from the Foundry client
# This is what we will use to send questions and receive answers
chat_client = foundry_client.get_openai_client()

## Step 5: Configure the MCP Tool

An **MCP tool** tells the agent how to connect to the external MCP server. We provide:
- A label (a friendly name for logging and debugging)
- The server URL (where the MCP server lives)
- An approval setting (`"never"` means the agent can use it freely without asking)

The Microsoft Learn MCP server is a **public endpoint**, so no project connection or authentication is needed. For private MCP servers that require authentication, you would also pass a `project_connection_id`.

In [ ]:
# Create the MCP tool configuration
learn_tool = MCPTool(
    server_label="microsoft_learn_server",           # A human-readable label for this tool
    server_url="https://learn.microsoft.com/api/mcp", # The URL of the Microsoft Learn MCP server
    require_approval="never",                         # Let the agent use this tool without asking permission
)

## Step 6: Build an MCP-Enabled Agent

Now we create our agent and attach the MCP tool to it. The agent's **instructions** tell it what kind of assistant it should be. The **tools** list gives it the ability to reach out to external services.

In [ ]:
# Create the agent with MCP tool attached
learn_agent = foundry_client.agents.create_version(
    agent_name="learning-resource-agent",   # A unique name to identify this agent
    definition=PromptAgentDefinition(
        model=my_model,                      # The AI model that powers the agent's thinking
        instructions=(
            "You are an educational assistant that uses the Microsoft Learn MCP server "
            "to help users find relevant tutorials, documentation, and learning paths."
        ),
        tools=[learn_tool],                  # Give the agent access to our MCP tool
    )
)

# Confirm the agent was created successfully
print(f"MCP-enabled agent created! ID: {learn_agent.id}")

## Step 7: Open a Chat Session

A **conversation** (also called a chat session) keeps track of the back-and-forth between you and the agent. Creating one gives us a unique session ID that ties all messages together.

In [ ]:
# Open a new conversation (chat session) with the agent
chat_session = chat_client.conversations.create()

# Print the session ID so we can reference it later
print(f"Chat session started! Session ID: {chat_session.id}")

## Step 8: Ask the Agent to Find Learning Resources

Finally, we send a question to our agent. Behind the scenes, the agent will use the MCP tool to query Microsoft Learn and return helpful results. Let's ask it about Azure learning paths!

In [ ]:
# Write the question we want to ask the agent
learning_question = "Find me beginner-friendly learning paths for getting started with Azure cloud services."

In [ ]:
# Send the question to the agent and get a response
# - conversation: links this message to our chat session
# - extra_body: tells the API to route this to our named agent
# - input: the actual question text
agent_response = chat_client.responses.create(
    conversation=chat_session.id,       # Use the chat session we created above
    extra_body={
        "agent": {
            "name": "learning-resource-agent",  # The name of the agent to use
            "type": "agent_reference"            # Tells the API this is an agent reference
        }
    },
    input=learning_question              # The question we want answered
)

# Print the agent's answer
print(f"Agent Response: {agent_response.output_text}")